# Notebook 03 — WAOUH (bonus)

Ce notebook contient 3 choses qui te démarquent :
1) **ROC/AUC** + **PR Curve** (si dispo)
2) **Threshold tuning** (choisir un seuil optimisé sur validation)
3) **Leakage audit** (montrer la différence avec/sans mots de genre)


## 1) Exécuter le pipeline (si pas fait)

In [ ]:
!python -m src.pipeline.prepare_data
!python -m src.pipeline.train
!python -m src.pipeline.evaluate

## 2) Lire les métriques et afficher les plots

In [ ]:
import json
from pathlib import Path
metrics = json.loads(Path('reports/metrics.json').read_text(encoding='utf-8'))
metrics.keys()

In [ ]:
metrics['test_default']['accuracy'], metrics['test_default']['f1_macro'], metrics.get('roc_auc')

### Plots générés

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

for p in ['reports/plots/confusion_matrix.png','reports/plots/roc_curve.png','reports/plots/pr_curve.png']:
    if Path(p).exists():
        img = Image.open(p)
        plt.figure()
        plt.imshow(img)
        plt.axis('off')
        plt.title(p)
        plt.show()

## 3) Leakage audit (mini expérience)

On compare une baseline **naïve** (texte brut) vs la version **corrigée** (suppression des tokens de genre).

> Objectif : montrer au prof qu’on a identifié et corrigé le risque de *target leakage*.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from src.text_utils import build_model_text

raw = pd.read_csv('data/raw/styles.csv', sep=';', on_bad_lines='skip')
raw.columns = raw.columns.str.strip()

def norm(g):
    if g in ['Men','Boys']: return 'MEN'
    if g in ['Women','Girls']: return 'WOMEN'
    return None

raw['gender_group'] = raw['gender'].apply(norm)
raw = raw.dropna(subset=['gender_group'])
for c in ['productDisplayName','subCategory','articleType']:
    raw[c] = raw[c].fillna('')

raw['text_naive'] = raw['productDisplayName'].astype(str)+' '+raw['subCategory'].astype(str)+' '+raw['articleType'].astype(str)
raw['text_clean'] = raw.apply(lambda r: build_model_text(r['productDisplayName'], r['subCategory'], r['articleType']), axis=1)

Xn, Xc, y = raw['text_naive'].astype(str), raw['text_clean'].astype(str), raw['gender_group'].astype(str)
Xn_tr, Xn_te, y_tr, y_te = train_test_split(Xn, y, test_size=0.2, random_state=42, stratify=y)
Xc_tr, Xc_te, _, _ = train_test_split(Xc, y, test_size=0.2, random_state=42, stratify=y)

pipe = Pipeline([('tfidf', TfidfVectorizer(stop_words='english')), ('clf', LogisticRegression(max_iter=2000))])

pipe.fit(Xn_tr, y_tr)
pred_n = pipe.predict(Xn_te)
f1_n = f1_score(y_te, pred_n, average='macro')

pipe.fit(Xc_tr, y_tr)
pred_c = pipe.predict(Xc_te)
f1_c = f1_score(y_te, pred_c, average='macro')

print('Macro-F1 naive (possible leakage):', round(float(f1_n),4))
print('Macro-F1 cleaned (no leakage tokens):', round(float(f1_c),4))

## Comment en parler dans le rapport
- "On a détecté un risque de leakage car *productDisplayName* contient parfois *Men/Women*"
- "On a fait une comparaison naive vs cleaned"
- "On retient la version cleaned pour le déploiement"
